In [15]:
# Celda 1 — imports
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, SILVER_DB, GOLD_DB

In [2]:
# Celda 2 — función genérica (misma lógica)
def cargar_a_gold(nombre_tabla, query):
    print(f"Leyendo silver para {nombre_tabla}...")
    result = CH.query(query)
    rows = result.result_rows
    cols = result.column_names

    df = pd.DataFrame(rows, columns=cols)

    df['_valid_from'] = datetime.now()
    df = df.where(pd.notnull(df), None)
    
    data = []
    for row in df.values.tolist():
        clean_row = []
        for val in row:
            if isinstance(val, float) and np.isnan(val):
                clean_row.append(None)
            elif isinstance(val, (np.integer,)):
                clean_row.append(int(val))
            elif isinstance(val, (np.floating,)):
                clean_row.append(float(val))
            else:
                clean_row.append(val)
        data.append(clean_row)


    CH.command(f"TRUNCATE TABLE {GOLD_DB}.{nombre_tabla}")
    CH.insert(f"{GOLD_DB}.{nombre_tabla}",
              data,
              column_names=df.columns.tolist())
    print(f"✓ {len(df):,} filas cargadas en gold.{nombre_tabla}")

In [22]:
gold_tablas = {

    # ==================== DIMENSIONES ====================

    'dim_customer': """
        SELECT
            c.customer_id                   AS customer_key,
            c.customer_id                   AS customer_id,
            c.first_name                    AS first_name,
            c.last_name                     AS last_name,
            c.email                         AS email,
            c.active                        AS active,
            a.address                       AS address,
            a.district                      AS district,
            a.postal_code                   AS postal_code,
            a.phone                         AS phone,
            ci.city                         AS city,
            co.country                      AS country
        FROM silver.customer c
        JOIN silver.address a   ON c.address_id = a.address_id
        JOIN silver.city ci     ON a.city_id = ci.city_id
        JOIN silver.country co  ON ci.country_id = co.country_id
    """,

    'dim_film': """
        SELECT
            f.film_id                       AS film_key,
            f.film_id                       AS film_id,
            f.title                         AS title,
            f.description                   AS description,
            f.release_year                  AS release_year,
            l.name                          AS language_name,
            COALESCE(ol.name, 'N/A')       AS original_language_name,
            f.rental_duration               AS rental_duration,
            f.rental_rate                   AS rental_rate,
            f.length                        AS length,
            f.replacement_cost              AS replacement_cost,
            f.rating                        AS rating,
            f.special_features              AS special_features,
            COALESCE(cat.name, 'N/A')      AS category_name
        FROM silver.film f
        JOIN silver.language l              ON f.language_id = l.language_id
        LEFT JOIN silver.language ol        ON f.original_language_id = ol.language_id
        LEFT JOIN silver.film_category fc   ON f.film_id = fc.film_id
        LEFT JOIN silver.category cat       ON fc.category_id = cat.category_id
    """,

    'dim_actor': """
        SELECT
            actor_id                        AS actor_key,
            actor_id                        AS actor_id,
            first_name                      AS first_name,
            last_name                       AS last_name
        FROM silver.actor
    """,

    'dim_store': """
        SELECT
            s.store_id                      AS store_key,
            s.store_id                      AS store_id,
            a.address                       AS address,
            a.district                      AS district,
            a.postal_code                   AS postal_code,
            a.phone                         AS phone,
            ci.city                         AS city,
            co.country                      AS country,
            st.first_name                   AS manager_first_name,
            st.last_name                    AS manager_last_name
        FROM silver.store s
        JOIN silver.address a   ON s.address_id = a.address_id
        JOIN silver.city ci     ON a.city_id = ci.city_id
        JOIN silver.country co  ON ci.country_id = co.country_id
        JOIN silver.staff st    ON s.manager_staff_id = st.staff_id
    """,

    'dim_staff': """
        SELECT
            st.staff_id                     AS staff_key,
            st.staff_id                     AS staff_id,
            st.first_name                   AS first_name,
            st.last_name                    AS last_name,
            st.email                        AS email,
            st.username                     AS username,
            st.active                       AS active,
            a.address                       AS address,
            a.district                      AS district,
            ci.city                         AS city,
            co.country                      AS country
        FROM silver.staff st
        JOIN silver.address a   ON st.address_id = a.address_id
        JOIN silver.city ci     ON a.city_id = ci.city_id
        JOIN silver.country co  ON ci.country_id = co.country_id
    """,

    'dim_date': """
        SELECT
            toInt32(formatDateTime(date, '%Y%m%d'))   AS date_key,
            date                                          AS full_date,
            toYear(date)                                  AS year,
            toMonth(date)                                 AS month,
            formatDateTime(date, '%M')                   AS month_name,
            toQuarter(date)                               AS quarter,
            toDayOfWeek(date)                             AS day_of_week,
            formatDateTime(date, '%W')                   AS day_name,
            if(toDayOfWeek(date) IN (6, 7), 1, 0)        AS is_weekend
        FROM (
            SELECT toDate('2005-01-01') + number AS date
            FROM numbers(dateDiff('day', toDate('2005-01-01'), toDate('2025-12-31')) + 1)
        )
    """,

    # ==================== FACT TABLES ====================

    'fact_film_actor': """
        SELECT
            fa.film_id                      AS film_key,
            fa.actor_id                     AS actor_key
        FROM silver.film_actor fa
    """,

    'fact_rental_payment': """
        SELECT
            r.rental_id                                                 AS rental_id,
            p.payment_id                                                AS payment_id,
            r.customer_id                                               AS customer_key,
            i.film_id                                                   AS film_key,
            i.store_id                                                  AS store_key,
            r.staff_id                                                  AS staff_key,
            toInt32(formatDateTime(r.rental_date, '%Y%m%d'))         AS rental_date_key,
            if(r.return_date IS NOT NULL,
                toInt32(formatDateTime(r.return_date, '%Y%m%d')),
                0)                                                      AS return_date_key,
            toInt32(formatDateTime(p.payment_date, '%Y%m%d'))        AS payment_date_key,
            i.inventory_id                                              AS inventory_id,
            r.rental_date                                               AS rental_date,
            r.return_date                                               AS return_date,
            p.payment_date                                              AS payment_date,
            p.amount                                                    AS amount,
            if(r.return_date IS NOT NULL,
                dateDiff('day', r.rental_date, r.return_date),
                0)                                                      AS rental_duration_actual,
            if(r.return_date IS NOT NULL
                AND dateDiff('day', r.rental_date, r.return_date) > f.rental_duration,
                1, 0)                                                   AS is_overdue,
            if(r.return_date IS NOT NULL, 1, 0)                         AS is_returned
        FROM silver.rental r
        JOIN silver.payment p       ON r.rental_id = p.rental_id
        JOIN silver.inventory i     ON r.inventory_id = i.inventory_id
        JOIN silver.film f          ON i.film_id = f.film_id
    """,
}

In [11]:
# Reemplazar la query de dim_date en el diccionario
gold_tablas['dim_date'] = None  # no usa query SQL

# Y agregar una función especial para dim_date
def cargar_dim_date():
    print("Generando dim_date...")
    dates = pd.date_range('1990-01-01', '2030-12-31', freq='D')

    df = pd.DataFrame({
        'date_key':     dates.strftime('%Y%m%d').astype(int),
        'full_date':    dates.date,
        'year':         dates.year.astype(int),
        'quarter':      dates.quarter.astype(int),
        'quarter_name': ('Q' + dates.quarter.astype(str)),
        'month':        dates.month.astype(int),
        'month_name':   dates.strftime('%B'),
        'week':         dates.isocalendar().week.astype(int),
        'day':          dates.day.astype(int),
        'day_name':     dates.strftime('%A'),
        'is_weekend':   (dates.weekday >= 5).astype(int),
    })

    CH.command(f"TRUNCATE TABLE {GOLD_DB}.dim_date")
    CH.insert(f"{GOLD_DB}.dim_date",
              df.values.tolist(),
              column_names=df.columns.tolist())
    print(f"✓ {len(df):,} filas cargadas en gold.dim_date")

In [13]:
cargar_dim_date()

Generando dim_date...
✓ 14,975 filas cargadas en gold.dim_date


In [23]:
# Celda 4 — ejecutar dims primero, fact al final
dims  = ['dim_staff','dim_store','dim_customer',
         'dim_film','dim_actor']
facts = ['fact_rental_payment', 'fact_film_actor']

for tabla in dims + facts:
    cargar_a_gold(tabla, gold_tablas[tabla])

Leyendo silver para dim_staff...
✓ 2 filas cargadas en gold.dim_staff
Leyendo silver para dim_store...
✓ 2 filas cargadas en gold.dim_store
Leyendo silver para dim_customer...
✓ 599 filas cargadas en gold.dim_customer
Leyendo silver para dim_film...
✓ 1,000 filas cargadas en gold.dim_film
Leyendo silver para dim_actor...
✓ 200 filas cargadas en gold.dim_actor
Leyendo silver para fact_rental_payment...
✓ 14,881 filas cargadas en gold.fact_rental_payment
Leyendo silver para fact_film_actor...
✓ 5,462 filas cargadas en gold.fact_film_actor
